# LogSeer Token Importance

Shows which tokens most influenced the XGBoost model's predictions, ranked by feature importance.

If you already ran training on this machine, `TOKENIZER_FILE` and `XGB_FILE` are already generated. Otherwise use the Colab cell to load them from Google Drive, or manually coping them to `MODEL_BASE`.

In [ ]:
# Setup
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

In [ ]:
# Configuration
MODEL_BASE     = './'                       # directory where trained models are stored
TOKENIZER_FILE = 'tokenizer.pickle'
XGB_FILE       = 'xgb.pkl'
TOKENIZER_PATH = MODEL_BASE + TOKENIZER_FILE
XGB_FILE_PATH  = MODEL_BASE + XGB_FILE
TOP_N          = 50

**Colab only** — Run below to copy trained models from Google Drive. Otherwise skip this cell.

In [ ]:
# Copy trained models from Google Drive.
from google.colab import drive
import shutil
drive.mount('/content/drive')

DRIVE_BASE      = '/content/drive/MyDrive/Colab Notebooks/logseer/'
DRIVE_MODEL_DIR = DRIVE_BASE + '/models/'

shutil.copy(DRIVE_MODEL_DIR + TOKENIZER_FILE, MODEL_BASE + TOKENIZER_FILE)
shutil.copy(DRIVE_MODEL_DIR + XGB_FILE,       MODEL_BASE + XGB_FILE)

In [ ]:
with open(MODEL_BASE + TOKENIZER_FILE, 'rb') as f:
    tokenizer = pickle.load(f)

with open(MODEL_BASE + XGB_FILE, 'rb') as f:
    model = pickle.load(f)

# word_index is token -> int, invert to int -> token
index_to_word = {v: k for k, v in tokenizer.word_index.items()}
print(f'Vocabulary size: {len(index_to_word)}')

In [ ]:
scores = model.feature_importances_

# texts_to_matrix columns correspond to token indices 0..MAX_NB_WORDS
# index 0 is unused padding, so token i maps to column i
rows = []
for col_idx, score in enumerate(scores):
    if score > 0 and col_idx in index_to_word:
        rows.append({'token': index_to_word[col_idx], 'importance': score})

df = pd.DataFrame(rows).sort_values('importance', ascending=False).reset_index(drop=True)
print(f'Non-zero features: {len(df)}')
df.head(TOP_N)

In [ ]:
top = df.head(TOP_N)
plt.figure(figsize=(10, TOP_N * 0.3))
plt.barh(top['token'][::-1], top['importance'][::-1])
plt.xlabel('Importance')
plt.title(f'Top {TOP_N} tokens by XGB feature importance')
plt.tight_layout()
plt.show()